# 🧭 RAG Variants — Simple Workflow Edition

**Aim:** har RAG variant ka **flow** samajhna. Is liye har variant mein:
1. 🧠 **Ek line mein** kya hai
2. 🗺️ **Workflow diagram** — `★` wala qadam **naya** hai (aap ke basic workflow se farq)
3. 🧪 **Code** — har step alag likha hai (`STEP 1`, `STEP 2` ...) taake flow saaf nazar aaye. Code thora repeat hoga, ye jaan-boojh kar hai.
4. 📌 **Yaad rakho** — kab use karna hai + interview line

---
## 🧱 Aap ka workflow = sab variants ki bunyaad

```
INDEXING  (ek dafa)
  Documents ──► Chunking ──► Embedding ──► Vector Store  (vectors + metadata)

QUERY  (har sawal par)
  Query ──► Rewrite ──► Query Embedding ──► Vector Search (top 5) ──► Rerank (top 3) ──► LLM ──► Answer
                                                                                   ▲
                                                               query + chunks + metadata
```

Har variant isi pipeline mein **sirf ek do cheezen add ya change** karta hai:

| # | Variant | Pipeline mein kya badla | Kis masle ka ilaj |
|---|---|---|---|
| 1 | **Naive RAG** | Rewrite aur Rerank **hata do** | Baseline (sab se simple) |
| 2 | **Advanced RAG** | Rewrite + Rerank (= **aap ka workflow**) | Ghalat/kamzor retrieval |
| 3 | **Self-RAG** | ★ "Retrieve karun?" + ★ chunks grade + ★ answer check | Faltu retrieval, hallucination |
| 4 | **CRAG** | ★ Evidence ko score karo → kamzor ho to ★ web search | Docs mein jawab hi nahi |
| 5 | **Iterative RAG** | Search → ★ "kaafi hai?" → dobara search (loop) | Missing info pehle se pata nahi |
| 6 | **Multi-hop RAG** | ★ Sawal ko sub-questions mein todo, har ek par pipeline | Jawab kayi docs mein bikhra |
| 7 | **Graph RAG** | Vector store ki jagah ★ knowledge graph | Entities ke rishte (A→B→C) |
| 8 | **Adaptive RAG** | Shuru mein ★ Router → sahi pipeline chuno | Har sawal par heavy pipeline (cost) |

> **Cells upar se neeche order mein chalao.** Har variant pehle wale common functions use karta hai.

---
# 0️⃣ Setup

**Groq API key** (free): console.groq.com → API Keys

**Option A — Google Colab:** left side 🔑 **Secrets** → naam `GROQ_API_KEY`, value apni key → "Notebook access" ON.

**Option B — Windows (local Jupyter / VS Code):** notebook wale folder mein `.env` file banao:
```
GROQ_API_KEY=gsk_xxxxxxxxxxxxxxxx
```

⚠️ Naam bilkul **`GROQ_API_KEY`** ho (sab capital). `Groq_API_KEY` likhoge to `KeyError` aayega.

In [ ]:
# %pip install -q groq python-dotenv sentence-transformers networkx ddgs

### 🔌 LLM helpers
- `llm(prompt)` → normal text jawab
- `llm_json(prompt)` → JSON jawab (graders, routers ke liye)
- `MODEL` = bara model (answer likhne ke liye), `FAST_MODEL` = chhota/sasta model (grading, routing ke liye)

In [4]:
import os, re, json
import numpy as np
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# ─────────────── API key ───────────────
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
# key
# ─────────────── Models ───────────────
MODEL = "openai/gpt-oss-120b"        # bara model: answer likhne ke liye
FAST_MODEL = "openai/gpt-oss-20b"    # chhota model: grading / routing ke liye
# Model error aaye (decommissioned / not found) to console.groq.com/docs/models se naya naam likho.

# LangChain ChatGroq — do model objects, ek dafa bana lo
big_llm  = ChatGroq(model=MODEL,      api_key=api_key, temperature=0, reasoning_effort="low", max_retries=5)
fast_llm = ChatGroq(model=FAST_MODEL, api_key=api_key, temperature=0, reasoning_effort="low", max_retries=5)

def pick(model):
    return big_llm if model == MODEL else fast_llm

def clean(text):
    # Agar kabhi model apni "thinking" <think>...</think> mein bhej de, to usey hata do
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

# ─────────────── LLM helpers ───────────────
def llm(prompt, model=MODEL):
    """Normal text jawab."""
    response = pick(model).invoke(prompt)
    return clean(response.content)

def llm_json(prompt, model=FAST_MODEL):
    """JSON jawab (graders, routers ke liye). Kharab JSON aaye to {} — code crash na ho."""
    json_llm = pick(model).bind(response_format={"type": "json_object"})   # JSON mode ON
    response = json_llm.invoke(prompt + "\n\nReply ONLY with valid JSON.")
    try:
        return json.loads(clean(response.content))
    except json.JSONDecodeError:
        return {}

# ─────────────── Safe conversion ───────────────
# LLM kabhi true ki jagah "true"/"yes" likh deta hai
def to_bool(x):
    if isinstance(x, bool): return x
    return str(x).strip().lower() in ("true", "yes", "1")

def to_float(x):
    try: return float(x)
    except (TypeError, ValueError): return 0.0

# ─────────────── Printing helpers (flow dikhane ke liye) ───────────────
def title(name, query):
    print("━" * 90)
    print(f"🔹 {name}\n❓ {query}")

def step(msg):
    print(f"\n▶ {msg}")

def show_hits(hits):
    for h in hits:
        score = h.get("rerank_score", h.get("score", 0))
        print(f"     • {h['chunk_id']:<26} {score:6.3f} | {h['text'][:70]}...")

print(llm("Say 'setup ok' and nothing else."))

setup ok


---
# 🧱 Common Building Blocks (aap ka workflow, step by step)

Ye 7 functions **har variant** mein use honge. Inhein ek dafa samajh lo, phir variants mein sirf naye qadam dekhne hain.

```
INDEXING:  [1] Data ingestion ─► [2] Chunking ─► [3] Embedding ─► [4] Vector Store
QUERY:     [5] Query rewrite ─► (embedding) ─► (search) ─► [6] Rerank ─► [7] LLM answer
```

## [1] Data Ingestion
NovaTech ek **fictional** company hai, is liye LLM apni memory se jawab nahi de sakta. Jo sahi jawab aaye ga, woh retrieval ki wajah se aaye ga.

Docs mein jaan-boojh kar ye traps hain:
- `hr-leave` (current) aur `hr-leave-old` (**archived**) → purani info ka trap
- `proj-falcon` → `people-ayesha` → `office-karachi` → **chain** (multi-hop / graph ke liye)

In [5]:
DOCS = [
    {"id": "hr-leave", "title": "Leave Policy", "status": "current",
     "text": "NovaTech employees get 24 days of paid annual leave per year. Up to 6 unused days can be carried forward to the next year. Sick leave is separate: 10 days per year, and a medical certificate is needed for more than 2 sick days in a row."},
    {"id": "hr-leave-old", "title": "Old Leave Policy (2022)", "status": "archived",
     "text": "ARCHIVED 2022 policy: employees received 18 days of annual leave per year and unused leave could not be carried forward."},
    {"id": "hr-remote", "title": "Remote Work Policy", "status": "current",
     "text": "Employees may work remotely up to 2 days per week after their 3-month probation. Fully remote work needs approval from the department head and the HR director, Sana Mirza."},
    {"id": "proj-falcon", "title": "Project Falcon", "status": "current",
     "text": "Project Falcon is NovaTech's AI customer-support assistant for retail clients. It is led by Ayesha Khan. Falcon's target launch is March 2027."},
    {"id": "proj-orion", "title": "Project Orion", "status": "current",
     "text": "Project Orion is an internal analytics dashboard. It is led by Hamza Riaz and is built with Python and Streamlit."},
    {"id": "people-ayesha", "title": "Ayesha Khan", "status": "current",
     "text": "Ayesha Khan is a senior machine learning engineer. She works from the Karachi office and reports to the CTO, Omar Farooq."},
    {"id": "people-hamza", "title": "Hamza Riaz", "status": "current",
     "text": "Hamza Riaz is a data engineer on the Data Platform team. He works from the Lahore office."},
    {"id": "office-karachi", "title": "Karachi Office", "status": "current",
     "text": "The Karachi office is on Shahrah-e-Faisal and is managed by Bilal Ahmed. It hosts the machine learning and customer-success teams."},
    {"id": "office-lahore", "title": "Lahore Office", "status": "current",
     "text": "The Lahore office in Gulberg is NovaTech's headquarters. It is managed by Fatima Noor and hosts engineering, HR and finance."},
    {"id": "product-novabot", "title": "NovaBot Pricing", "status": "current",
     "text": "NovaBot, NovaTech's chatbot, costs USD 49 per month on the Starter plan and USD 199 per month on the Business plan. The Business plan includes priority support and 10,000 conversations per month."},
    {"id": "product-refund", "title": "Refund Policy", "status": "current",
     "text": "Customers can get a full refund within 14 days of purchase. After 14 days, refunds are given only for annual plans and are pro-rated."},
]
print(len(DOCS), "documents loaded")

11 documents loaded


## [2] Chunking
Sentences ko jodte jao jab tak `max_chars` na ho jaye. Har chunk ke saath **metadata** (source, title, status) rakho.

**Chunk header trick:** har chunk ke shuru mein doc ka title laga do (`"Project Falcon: It is led by Ayesha Khan."`). Warna "It" kis ki baat kar raha hai, ye chunk akele mein pata nahi chalta, aur search fail hoti hai.

In [6]:
def chunk_text(text, max_chars=200):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for s in sentences:
        if current and len(current) + len(s) + 1 > max_chars:
            chunks.append(current)
            current = s
        else:
            current = (current + " " + s).strip()
    if current:
        chunks.append(current)
    return chunks

all_chunks = []
for doc in DOCS:
    for i, piece in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "chunk_id": f'{doc["id"]}#{i}',
            "text": f'{doc["title"]}: {piece}',            # chunk header trick
            "metadata": {"source": doc["id"], "title": doc["title"], "status": doc["status"]},
        })

print(len(DOCS), "docs ->", len(all_chunks), "chunks\n")
for c in all_chunks[:4]:
    print(c["chunk_id"], "|", c["text"])

11 docs -> 12 chunks

hr-leave#0 | Leave Policy: NovaTech employees get 24 days of paid annual leave per year. Up to 6 unused days can be carried forward to the next year.
hr-leave#1 | Leave Policy: Sick leave is separate: 10 days per year, and a medical certificate is needed for more than 2 sick days in a row.
hr-leave-old#0 | Old Leave Policy (2022): ARCHIVED 2022 policy: employees received 18 days of annual leave per year and unused leave could not be carried forward.
hr-remote#0 | Remote Work Policy: Employees may work remotely up to 2 days per week after their 3-month probation. Fully remote work needs approval from the department head and the HR director, Sana Mirza.


## [3] Embedding
`all-MiniLM-L6-v2` (free, local). Text → 384 numbers ka vector. Milte-julte matlab wale texts ke vectors paas paas hote hain.

In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

v = embed(["annual leave days", "how many holidays do employees get", "laptop price"])
print("shape:", v.shape)
print("leave vs holidays:", round(float(v[0] @ v[1]), 3))
print("leave vs laptop  :", round(float(v[0] @ v[2]), 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

shape: (3, 384)
leave vs holidays: 0.544
leave vs laptop  : -0.008


## [4] Vector Store
Sab se simple vector DB: saare vectors ek numpy matrix mein. Search = query vector ka har chunk vector se **dot product** (normalized vectors par ye cosine similarity hai), phir top-k.

> Production mein yahi kaam **Chroma / FAISS / Qdrant / Pinecone** karte hain, bas tez aur bare scale par.

In [8]:
class SimpleVectorStore:
    def __init__(self):
        self.chunks = []
        self.vectors = None

    def add(self, chunks, vectors):
        self.chunks.extend(chunks)
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])

    def search(self, query_vector, top_k=5):
        scores = self.vectors @ query_vector
        best = np.argsort(-scores)[:top_k]
        return [{**self.chunks[i], "score": float(scores[i])} for i in best]

# INDEXING: chunks ─► embedding ─► vector store
vector_store = SimpleVectorStore()
vector_store.add(all_chunks, embed([c["text"] for c in all_chunks]))
print("Vector store ready:", vector_store.vectors.shape)

Vector store ready: (12, 384)


## [5] Query Rewrite
User ki query ko **search ke liye behtar** banao: English mein, saaf, keywords ke saath (Roman Urdu wale sawal ke liye zaroori).

In [9]:
def rewrite_query(query):
    prompt = f"""Rewrite this question as a short, clear English search query for a company knowledge base.
Translate to English if needed. Keep names. Return ONLY the query.

Question: {query}"""
    return llm(prompt, model=FAST_MODEL).strip('"')

print(rewrite_query("saal mein kitni chuttiyan milti hain?"))

How many holidays are given in a year?


## [6] Rerank (Cross-Encoder)
Vector search tez hai magar query aur chunk ko **alag alag** dekhta hai. Cross-encoder dono ko **ek saath** padh kar zyada sahi score deta hai, magar slow hai. Is liye: **top 5 lo → rerank → top 3 rakho**.

In [10]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, hits, top_n=3):
    if not hits:
        return []
    scores = reranker.predict([(query, h["text"]) for h in hits])
    for h, s in zip(hits, scores):
        h["rerank_score"] = float(s)
    return sorted(hits, key=lambda h: h["rerank_score"], reverse=True)[:top_n]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## [7] LLM Answer (query + chunks + metadata)
Context mein har chunk ke upar uska metadata likha hota hai. Prompt LLM ko kehta hai: sirf context se jawab do, archived ignore karo, source cite karo.

In [11]:
def build_context(hits):
    parts = []
    for h in hits:
        m = h["metadata"]
        parts.append(f"[source: {m['source']} | title: {m['title']} | status: {m['status']}]\n{h['text']}")
    return "\n\n".join(parts)

def generate_answer(query, hits):
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the documents."
Ignore documents with status 'archived' when a current one exists.
Cite sources like [source].

Context:
{build_context(hits)}

Question: {query}
Answer:"""
    return llm(prompt)

### 🧪 Test sawal (har variant inhi par chalega)

In [12]:
Q_SIMPLE   = "How many annual leave days do employees get?"
Q_URDU     = "saal mein kitni chuttiyan milti hain aur kitni agle saal le ja sakte hain?"
Q_MULTIHOP = "Who manages the office where the lead of Project Falcon works?"
Q_OUTSIDE  = "What is the latest stable version of Python?"
Q_GENERAL  = "What is 15% of 240?"

---
# 1️⃣ Naive RAG

### 🧠 Ek line mein
Query ko embed karo → top-k chunks lo → LLM ko do. **Koi rewrite nahi, koi rerank nahi, koi check nahi.**

### 🗺️ Workflow
```
Query ──► Query Embedding ──► Vector Search (top 3) ──► LLM ──► Answer

(aap ke workflow se Rewrite aur Rerank HATA diye)
```

In [13]:
def naive_rag(query):
    title("NAIVE RAG", query)

    step("STEP 1: Query embedding")
    q_vec = embed([query])[0]

    step("STEP 2: Vector search (top 3)")
    hits = vector_store.search(q_vec, top_k=3)
    show_hits(hits)

    step("STEP 3: LLM (query + chunks + metadata)")
    answer = generate_answer(query, hits)
    print("\n🤖", answer)
    return answer

naive_rag(Q_SIMPLE)
naive_rag(Q_URDU)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 NAIVE RAG
❓ How many annual leave days do employees get?

▶ STEP 1: Query embedding

▶ STEP 2: Vector search (top 3)
     • hr-leave-old#0              0.700 | Old Leave Policy (2022): ARCHIVED 2022 policy: employees received 18 d...
     • hr-leave#0                  0.635 | Leave Policy: NovaTech employees get 24 days of paid annual leave per ...
     • hr-leave#1                  0.517 | Leave Policy: Sick leave is separate: 10 days per year, and a medical ...

▶ STEP 3: LLM (query + chunks + metadata)

🤖 Employees receive 24 days of paid annual leave per year【hr-leave】.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 NAIVE RAG
❓ saal mein kitni chuttiyan milti hain aur kitni agle saal le ja sakte hain?

▶ STEP 1: Query embedding

▶ STEP 2: Vector search (top 3)
     • people-ayesha#0             0.155 | Ayesha Khan: Ayesha Khan is a senior machin

"I don't know based on the documents."

### 📌 Yaad rakho
- Ye **baseline** hai. Pehle isey chalao, dekho kahan fail hota hai, phir variant chuno.
- Roman Urdu wale sawal par ghalat chunks aa sakte hain (embedding model English par trained hai), aur archived policy bhi aa sakti hai.
- **Interview:** "Naive RAG ki 3 kamzoriyan: single dense search, koi verification nahi, single-shot (multi-hop nahi kar sakta)."

---
# 2️⃣ Advanced RAG (= aap ka workflow)

### 🧠 Ek line mein
Naive pipeline ke **pehle** (query rewrite) aur **baad** (rerank) behtari layers lagao. Flow ab bhi seedha line mein hai.

### 🗺️ Workflow
```
Query ──► ★Rewrite ──► Query Embedding ──► Vector Search (top 5) ──► ★Rerank (top 3) ──► LLM ──► Answer
          pre-retrieval                                                post-retrieval
```

**Aur bhi Advanced RAG tricks** (isi family mein, code yahan nahi):
| Trick | Kya karti hai |
|---|---|
| Multi-query | Ek sawal ki 3 phrasings banao, teeno se search, results jodo |
| HyDE | LLM se farzi jawab likhwao, **us** ko embed kar ke search karo |
| Hybrid search | BM25 (keyword) + vector search, phir RRF se jodo — naam/IDs ke liye zabardast |
| Metadata filter | Search se pehle `status = current` jaisa filter |

In [14]:
def advanced_rag(query):
    title("ADVANCED RAG", query)

    step("STEP 1: ★ Query rewrite")
    new_query = rewrite_query(query)
    print("     ", new_query)

    step("STEP 2: Query embedding")
    q_vec = embed([new_query])[0]

    step("STEP 3: Vector search (top 5)")
    hits = vector_store.search(q_vec, top_k=5)
    show_hits(hits)

    step("STEP 4: ★ Rerank (top 3)")
    hits = rerank(new_query, hits, top_n=3)
    show_hits(hits)

    step("STEP 5: LLM (query + chunks + metadata)")
    answer = generate_answer(query, hits)
    print("\n🤖", answer)
    return answer

advanced_rag(Q_URDU)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 ADVANCED RAG
❓ saal mein kitni chuttiyan milti hain aur kitni agle saal le ja sakte hain?

▶ STEP 1: ★ Query rewrite
      Annual holidays count and carry‑over policy?

▶ STEP 2: Query embedding

▶ STEP 3: Vector search (top 5)
     • hr-leave-old#0              0.484 | Old Leave Policy (2022): ARCHIVED 2022 policy: employees received 18 d...
     • hr-leave#1                  0.350 | Leave Policy: Sick leave is separate: 10 days per year, and a medical ...
     • hr-leave#0                  0.295 | Leave Policy: NovaTech employees get 24 days of paid annual leave per ...
     • product-refund#0            0.174 | Refund Policy: Customers can get a full refund within 14 days of purch...
     • hr-remote#0                 0.078 | Remote Work Policy: Employees may work remotely up to 2 days per week ...

▶ STEP 4: ★ Rerank (top 3)
     • hr-leave-old#0             -3.671 | Old Leave Policy (2022)

'NovaTech में कर्मचारियों को साल में **24 दिन की वेतनभोगी वार्षिक छुट्टी** मिलती है, और उनमें से **अधिकतम 6 दिन** अनउपयोगित रहने पर अगले साल ले जाया जा सकता है【hr-leave】.'

### 📌 Yaad rakho
- Zyada tar production RAG systems yahi hain.
- Rerank se pehle **zyada** chunks lo (5–20), warna reranker ke paas chunne ko kuch nahi hoga.
- **Interview:** "Retrieve wide, rerank narrow. Bi-encoder tez hai (retrieval), cross-encoder accurate hai (reranking)."

---
# 3️⃣ Self-RAG

### 🧠 Ek line mein
Model **khud tay karta hai** ke retrieve karna hai ya nahi, phir har chunk ko **grade** karta hai, aur aakhir mein **apne jawab ko check** karta hai ke woh chunks se supported hai.

### 🗺️ Workflow
```
Query
  │
  ▼
🤖 LLM: [Retrieve?] "documents chahiye ya nahi?"
  │
  ├── no ──► 🤖 LLM: seedha answer de do ──► Answer ✅
  │
  ▼ yes
🤖 LLM (fast): Query rewrite
  │
  ▼
Embedding model: query ko vector banao          (LLM nahi)
  │
  ▼
Vector store: top 5 chunks                      (LLM nahi)
  │
  ▼
Cross-encoder: rerank → top 3                   (LLM nahi)
  │
  ▼
🤖 LLM (fast): [ISREL] har chunk relevant hai?   (3 chunks = 3 calls)
  │
  ├── koi relevant nahi ──► rewrite + dobara retrieve  (ya "I don't know")
  │
  ▼ relevant chunks
🤖 LLM (big): query + relevant chunks + metadata ──► answer likho
  │
  ▼
🤖 LLM (big): [ISSUP] answer chunks se supported hai?
  │
  ├── no ──► dobara try  (ya "I don't know")
  │
  ▼ yes
🤖 LLM (fast): [ISUSE] answer useful hai? (1–5)
  │
  ├── no ──► dobara try
  │
  ▼ yes
Answer ✅

```

Asli paper (Asai et al., 2023) mein model ko ye **reflection tokens** (`[Retrieve]`, `[ISREL]`, `[ISSUP]`, `[ISUSE]`) bolne ki training di gayi thi. Hum yahan har check ko ek alag LLM call se kar rahe hain (prompted version).

In [15]:
def self_rag(query):
    title("SELF-RAG", query)

    # ★ STEP A: retrieval chahiye ya nahi?
    step("STEP A: ★ [Retrieve?] — documents chahiye?")
    d = llm_json(f"""Does answering this question need NovaTech company documents (policies, people, projects, products, offices)?
General knowledge or math does NOT need documents.
Return JSON: {{"retrieve": true or false}}

Question: {query}""")
    need = to_bool(d.get("retrieve", True))
    print("      retrieve =", need)

    if not need:
        step("LLM seedha jawab (no retrieval)")
        answer = llm(query)
        print("\n🤖", answer)
        return answer

    # Normal pipeline (aap ka workflow)
    step("STEP 1: Query rewrite")
    new_query = rewrite_query(query)
    print("     ", new_query)

    step("STEP 2: Embedding + search (top 5)")
    q_vec = embed([new_query])[0]
    hits = vector_store.search(q_vec, top_k=5)

    step("STEP 3: Rerank (top 3)")
    hits = rerank(new_query, hits, top_n=3)
    show_hits(hits)

    # ★ STEP B: har chunk grade karo
    step("STEP B: ★ [ISREL] — har chunk relevant hai?")
    good_hits = []
    for h in hits:
        g = llm_json(f"""Is this document relevant AND current (not archived) for answering the question?
Return JSON: {{"relevant": true or false}}

Question: {query}
Document: {h['text']}""")
        ok = to_bool(g.get("relevant"))
        print(f"      {h['chunk_id']:<26} relevant = {ok}")
        if ok:
            good_hits.append(h)

    if not good_hits:
        answer = "I don't know based on the documents."
        print("\n🤖", answer)
        return answer

    step("STEP 4: LLM answer (sirf relevant chunks se)")
    answer = generate_answer(query, good_hits)
    print("      draft:", answer)

    # ★ STEP C: answer check
    step("STEP C: ★ [ISSUP] — answer chunks se supported hai?")
    s = llm_json(f"""Is every claim in the answer supported by the context?
Return JSON: {{"supported": true or false}}

Context:
{build_context(good_hits)}

Answer: {answer}""", model=MODEL)
    supported = to_bool(s.get("supported"))
    print("      supported =", supported)

    if not supported:
        answer = "I don't know based on the documents."
    print("\n🤖", answer)
    return answer

self_rag(Q_SIMPLE)
self_rag(Q_GENERAL)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 SELF-RAG
❓ How many annual leave days do employees get?

▶ STEP A: ★ [Retrieve?] — documents chahiye?
      retrieve = True

▶ STEP 1: Query rewrite
      Annual leave days employees receive?

▶ STEP 2: Embedding + search (top 5)

▶ STEP 3: Rerank (top 3)
     • hr-leave-old#0              7.892 | Old Leave Policy (2022): ARCHIVED 2022 policy: employees received 18 d...
     • hr-leave#0                  3.996 | Leave Policy: NovaTech employees get 24 days of paid annual leave per ...
     • hr-leave#1                 -1.281 | Leave Policy: Sick leave is separate: 10 days per year, and a medical ...

▶ STEP B: ★ [ISREL] — har chunk relevant hai?
      hr-leave-old#0             relevant = False
      hr-leave#0                 relevant = True
      hr-leave#1                 relevant = False

▶ STEP 4: LLM answer (sirf relevant chunks se)
      draft: Employees receive **24 days of paid annual 

'15\u202f% of 240 is **36**.'

### 📌 Yaad rakho
- **Mehenga hai:** ek sawal par 5–8 LLM calls. Sirf high-stakes jagah use karo (legal, medical, finance).
- Grading ke liye **alag aur sakht prompt** rakho, warna model apne hi jawab ko "supported" keh deta hai.
- **Interview:** "Self-RAG **generation** ko critique karta hai: kya mera jawab evidence se supported hai?"

---
# 4️⃣ CRAG — Corrective RAG

### 🧠 Ek line mein
Retrieve hue chunks ko ek **evaluator** se score karo. Evidence strong ho to use karo, kamzor ho to **web search** se correct karo.

### 🗺️ Workflow
```
Query ──► Rewrite ──► Embedding ──► Search (top 5) ──► Rerank (top 3)
                                                            │
                                                            ▼
                                     ★ Evaluator: har chunk ko score (0 to 1)
                                                            │
             ┌──────────────────────────────┬───────────────┴──────────────────┐
             ▼                              ▼                                  ▼
   ✅ CORRECT (best ≥ 0.7)      ⚠️ AMBIGUOUS (0.3 – 0.7)          ❌ INCORRECT (best < 0.3)
   internal chunks use karo     internal chunks + ★web search      internal phenko, ★web search
             └──────────────────────────────┴──────────────────────────────────┘
                                                            ▼
                                                    LLM ──► Answer
```

In [16]:
UPPER, LOWER = 0.7, 0.3

def web_search(query, n=3):
    """DuckDuckGo search. Results ko bhi chunk format mein badal do taake generate_answer() wahi rahe."""
    try:
        from ddgs import DDGS
        results = DDGS().text(query, max_results=n) or []
    except Exception as e:
        print("      ⚠️ web search fail:", e)
        return []
    return [{"chunk_id": f"web-{i}",
             "text": f"{r.get('title', '')}: {r.get('body', '')}",
             "metadata": {"source": r.get("href", "web"), "title": r.get("title", ""), "status": "web"}}
            for i, r in enumerate(results)]

def crag(query):
    title("CRAG (Corrective RAG)", query)

    step("STEP 1: Query rewrite")
    new_query = rewrite_query(query)
    print("     ", new_query)

    step("STEP 2: Embedding + search (top 5)")
    q_vec = embed([new_query])[0]
    hits = vector_store.search(q_vec, top_k=5)

    step("STEP 3: Rerank (top 3)")
    hits = rerank(new_query, hits, top_n=3)
    show_hits(hits)

    # ★ STEP 4: evaluator
    step("STEP 4: ★ Evaluator — har chunk ko score")
    for h in hits:
        g = llm_json(f"""How well does this document help answer the question?
Score from 0.0 (useless) to 1.0 (directly answers it). Archived/outdated documents get a low score.
Return JSON: {{"score": number}}

Question: {query}
Document: {h['text']}""")
        h["eval_score"] = max(0.0, min(1.0, to_float(g.get("score"))))
        print(f"      {h['chunk_id']:<26} score = {h['eval_score']:.2f}")

    best = max(h["eval_score"] for h in hits) if hits else 0.0
    if best >= UPPER:
        action = "CORRECT"
    elif best < LOWER:
        action = "INCORRECT"
    else:
        action = "AMBIGUOUS"
    print(f"\n      ➜ action = {action}  (best score {best:.2f})")

    # Action ke hisaab se knowledge banao
    if action == "INCORRECT":
        knowledge = []                                           # internal phenko
    else:
        knowledge = [h for h in hits if h["eval_score"] >= LOWER]  # achhe chunks rakho

    if action in ("INCORRECT", "AMBIGUOUS"):
        step("STEP 5: ★ Web search (corrective retrieval)")
        web_hits = web_search(new_query)
        show_hits(web_hits)
        knowledge += web_hits

    step("STEP 6: LLM")
    if not knowledge:
        answer = "I couldn't find reliable information for this."
    else:
        answer = generate_answer(query, knowledge)
    print("\n🤖", answer)
    return answer

crag("What does the NovaBot Business plan cost?")
crag(Q_OUTSIDE)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 CRAG (Corrective RAG)
❓ What does the NovaBot Business plan cost?

▶ STEP 1: Query rewrite
      NovaBot Business plan cost?

▶ STEP 2: Embedding + search (top 5)

▶ STEP 3: Rerank (top 3)
     • product-novabot#0           9.507 | NovaBot Pricing: NovaBot, NovaTech's chatbot, costs USD 49 per month o...
     • hr-leave#0                 -7.838 | Leave Policy: NovaTech employees get 24 days of paid annual leave per ...
     • office-lahore#0            -8.672 | Lahore Office: The Lahore office in Gulberg is NovaTech's headquarters...

▶ STEP 4: ★ Evaluator — har chunk ko score
      product-novabot#0          score = 1.00
      hr-leave#0                 score = 0.00
      office-lahore#0            score = 0.00

      ➜ action = CORRECT  (best score 1.00)

▶ STEP 6: LLM

🤖 The NovaBot Business plan costs **USD 199 per month**【source: product-novabot】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

'The latest stable version of Python is **Python\u202f3.13.2**【source】.'

### 📌 Yaad rakho
- Thresholds (0.7 / 0.3) apne data par chhote labelled set se **tune** karo.
- Company ke private sawal **web par mat bhejo** (data leak). Web results ka source hamesha dikhao.
- Sasta tareeqa: LLM grader ki jagah **reranker score** ko hi evaluator bana lo.
- **Interview:** "Self-RAG **jawab** ko check karta hai, CRAG **evidence** ko check aur correct karta hai."

---
# 5️⃣ Iterative RAG

### 🧠 Ek line mein
Search → LLM se poocho "evidence kaafi hai?" → nahi to LLM batata hai **kya missing hai** aur agli query banata hai → dobara search. Ye loop chalta rehta hai jab tak jawab na mile ya max rounds na ho jayein.

### 🗺️ Workflow
```
Query ──► Embedding ──► Search (top 3) ──► Evidence pool mein jodo
                ▲                                   │
                │                                   ▼
                │                   ★ LLM: "evidence kaafi hai?"
                │                          │                │
                │                         no               yes
                │                          ▼                ▼
                └──── ★ next_query ◄── missing kya hai?   Answer

(max 3 rounds — infinite loop se bachne ke liye)
```

In [17]:
def iterative_rag(query, max_rounds=3):
    title("ITERATIVE RAG", query)

    evidence = {}            # chunk_id -> chunk  (duplicate chunks na aayein)
    search_query = query

    for round_no in range(1, max_rounds + 1):
        print(f"\n── Round {round_no} ──  search query: {search_query}")

        step("STEP 1: Embedding + search (top 3)")
        q_vec = embed([search_query])[0]
        hits = vector_store.search(q_vec, top_k=3)
        for h in hits:
            evidence[h["chunk_id"]] = h
        print("      evidence pool:", list(evidence))

        step("STEP 2: ★ LLM — evidence kaafi hai?")
        d = llm_json(f"""Using ONLY the context, can the question be fully answered?
Return JSON: {{"enough": true or false, "answer": "...", "missing": "...", "next_query": "..."}}
If not enough, "next_query" must be a short search query for the missing fact.

Question: {query}

Context:
{build_context(list(evidence.values()))}""", model=MODEL)
        enough = to_bool(d.get("enough"))
        print(f"      enough = {enough} | missing = {d.get('missing', '')}")

        if enough and d.get("answer"):
            print("\n🤖", d["answer"])
            return d["answer"]

        # ★ STEP 3: agli query
        next_query = str(d.get("next_query", "")).strip()
        if not next_query or next_query == search_query:
            break
        search_query = next_query

    step("Max rounds khatam — jo evidence hai us se best answer")
    answer = generate_answer(query, list(evidence.values()))
    print("\n🤖", answer)
    return answer

iterative_rag(Q_MULTIHOP)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 ITERATIVE RAG
❓ Who manages the office where the lead of Project Falcon works?

── Round 1 ──  search query: Who manages the office where the lead of Project Falcon works?

▶ STEP 1: Embedding + search (top 3)
      evidence pool: ['proj-falcon#0', 'proj-orion#0', 'office-lahore#0']

▶ STEP 2: ★ LLM — evidence kaafi hai?
      enough = False | missing = The office location where Ayesha Khan, the lead of Project Falcon, works is not provided in the context.

── Round 2 ──  search query: Ayesha Khan NovaTech office location

▶ STEP 1: Embedding + search (top 3)
      evidence pool: ['proj-falcon#0', 'proj-orion#0', 'office-lahore#0', 'people-ayesha#0', 'office-karachi#0']

▶ STEP 2: ★ LLM — evidence kaafi hai?
      enough = True | missing = 

🤖 Bilal Ahmed manages the Karachi office where Ayesha Khan, the lead of Project Falcon, works.


'Bilal Ahmed manages the Karachi office where Ayesha Khan, the lead of Project Falcon, works.'

### 📌 Yaad rakho
- `max_rounds` **zaroor** rakho, aur same query dobara aaye to loop tod do.
- Evidence pool bara hota jata hai, context window bhar sakti hai. Dedupe karo, zaroorat ho to har round ke baad rerank.
- **Interview:** "Iterative ek **technique** hai (loop), multi-hop ek **problem type** hai. Iterative RAG multi-hop sawal hal karne ka ek tareeqa hai."

---
# 6️⃣ Multi-Hop RAG

### 🧠 Ek line mein
Jab jawab ke liye **kayi qadam** chahiye hon: pehle A dhoondo, us se B ka pata chale, phir B dhoondo. Sawal ko pehle hi **sub-questions** mein tod do, phir har ek par normal pipeline chalao.

**Example:** "Who manages the office where the lead of Project Falcon works?"
1. Project Falcon ka lead kaun? → **Ayesha Khan**
2. `#1` (Ayesha Khan) kis office mein? → **Karachi**
3. `#2` (Karachi) office ka manager? → **Bilal Ahmed**

### 🗺️ Workflow
```
Query ──► ★ Decompose (LLM sawal ko sub-questions mein todta hai)
              │
              ├─ Sub-Q 1 ──────────────────► [Embedding ► Search ► Rerank ► LLM] ──► answer 1
              ├─ Sub-Q 2 (#1 → answer 1) ──► [Embedding ► Search ► Rerank ► LLM] ──► answer 2
              └─ Sub-Q 3 (#2 → answer 2) ──► [Embedding ► Search ► Rerank ► LLM] ──► answer 3
                                                                                    │
                                        ★ Final LLM: saare steps jod kar ──► Answer ◄┘
```

In [18]:
def decompose(query):
    d = llm_json(f"""Break the question into the minimal list of simple sub-questions that must be answered IN ORDER.
Refer to the answer of an earlier sub-question as #1, #2, ...
Example: "What is the capital of the country where the inventor of X was born?"
-> ["Who invented X?", "In which country was #1 born?", "What is the capital of #2?"]
Return JSON: {{"sub_questions": ["...", "..."]}}

Question: {query}""", model=MODEL)
    subs = [s for s in d.get("sub_questions", []) if isinstance(s, str) and s.strip()]
    return subs or [query]

def multi_hop_rag(query):
    title("MULTI-HOP RAG", query)

    step("STEP 1: ★ Decompose")
    sub_questions = decompose(query)
    for i, sq in enumerate(sub_questions, 1):
        print(f"      {i}. {sq}")

    answers = []
    for i, sub_q in enumerate(sub_questions, 1):
        # #1, #2 ... ko pichle answers se replace karo
        for j in range(len(answers), 0, -1):
            sub_q = sub_q.replace(f"#{j}", answers[j - 1])
        print(f"\n── Hop {i}: {sub_q}")

        # Har hop par aap ka normal pipeline
        q_vec = embed([sub_q])[0]
        hits = vector_store.search(q_vec, top_k=5)
        hits = rerank(sub_q, hits, top_n=3)
        print("      top chunk:", hits[0]["chunk_id"])

        short_answer = llm(f"""Using ONLY the context, answer in a few words (just the name or fact).
If it is not in the context, say "unknown".

Context:
{build_context(hits)}

Question: {sub_q}""")
        answers.append(short_answer)
        print("      →", short_answer)

    step("STEP 2: ★ Final answer (saare hops jod kar)")
    steps_text = "\n".join(f"{i}. {q} -> {a}" for i, (q, a) in enumerate(zip(sub_questions, answers), 1))
    answer = llm(f"""Original question: {query}

Reasoning steps:
{steps_text}

Give the final answer in 1-2 sentences.""")
    print("\n🤖", answer)
    return answer

print("=== Pehle NAIVE (fail hoga) ===")
naive_rag(Q_MULTIHOP)
print("\n=== Ab MULTI-HOP ===")
multi_hop_rag(Q_MULTIHOP)

=== Pehle NAIVE (fail hoga) ===
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 NAIVE RAG
❓ Who manages the office where the lead of Project Falcon works?

▶ STEP 1: Query embedding

▶ STEP 2: Vector search (top 3)
     • proj-falcon#0               0.694 | Project Falcon: Project Falcon is NovaTech's AI customer-support assis...
     • proj-orion#0                0.373 | Project Orion: Project Orion is an internal analytics dashboard. It is...
     • office-lahore#0             0.329 | Lahore Office: The Lahore office in Gulberg is NovaTech's headquarters...

▶ STEP 3: LLM (query + chunks + metadata)

🤖 I don't know based on the documents.

=== Ab MULTI-HOP ===
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 MULTI-HOP RAG
❓ Who manages the office where the lead of Project Falcon works?

▶ STEP 1: ★ Decompose
      1. Who is the lead of Project Falcon?
      2. In which office does #1 work?
      

'The lead of Project\u202fFalcon, Ayesha\u202fKhan, works in the Karachi office, which is managed by Bilal\u202fAhmed.'

### 📌 Yaad rakho
- **Error propagation:** hop 1 ghalat → baaqi sab ghalat. "unknown" aaye to wahin ruk jao.
- Dependent sub-questions **order mein** chalao. Parallel sirf independent ke liye (jaise "X aur Y ka comparison").
- **Interview:** "Multi-hop ke 4 tareeqe: decomposition (upfront plan), iterative (dynamic), Graph RAG, agentic RAG."

---
# 7️⃣ Graph RAG

### 🧠 Ek line mein
Text se **entities** (log, projects, offices) aur unke **rishte** nikaal kar ek **knowledge graph** banao. Search chunks ke bajaye graph par karo. Rishton wale sawal graph mein bas "chal kar" mil jate hain.

### 🗺️ Workflow
```
INDEXING (ek dafa — mehenga, har chunk par 1 LLM call):
  Chunks ──► ★ LLM: (entity, relation, entity) triples nikaalo ──► ★ Knowledge Graph

          Ayesha Khan ──leads──► Project Falcon
          Ayesha Khan ──works_at──► Karachi Office
          Bilal Ahmed ──manages──► Karachi Office

QUERY:
  Query ──► ★ Entities dhoondo ──► ★ Graph mein woh nodes ──► ★ 3-hop padosi (subgraph)
                                                                     │
                                                    facts list ──► LLM ──► Answer
```

**Do tarah ki search:**
- **Local search** (yahan code hai): sawal ki entity ke aas paas ke facts. Multi-hop sawalon ke liye.
- **Global search** (Microsoft GraphRAG): graph ko communities (clusters) mein baanto, har ek ka summary likho, phir sab summaries se jawab jodo. "Poore data ke main themes kya hain?" jaise sawalon ke liye, jo top-k chunks se kabhi nahi milte.

In [ ]:
import networkx as nx

def extract_triples(text):
    d = llm_json(f"""Extract knowledge-graph triples from the text.
Use full canonical names (e.g. "Ayesha Khan", "Karachi Office", "Project Falcon").
Relations should be short verbs like leads, works_at, manages, reports_to, located_in, costs.
Return JSON: {{"triples": [{{"head": "...", "relation": "...", "tail": "..."}}]}}

Text: {text}""", model=MODEL)
    triples = []
    for t in d.get("triples", []) or []:
        if isinstance(t, dict) and t.get("head") and t.get("tail"):
            triples.append((str(t["head"]).strip(), str(t.get("relation", "related_to")).strip(), str(t["tail"]).strip()))
    return triples

# ★ INDEXING: chunks ─► triples ─► graph
graph = nx.DiGraph()
for c in all_chunks:
    if c["metadata"]["status"] == "archived":       # purani info graph mein mat daalo
        continue
    for head, rel, tail in extract_triples(c["text"]):
        graph.add_edge(head, tail, relation=rel, source=c["chunk_id"])

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges\n")
for u, v, data in list(graph.edges(data=True))[:12]:
    print(f"  {u} --{data['relation']}--> {v}")

In [ ]:
def graph_rag(query, hops=3):
    title("GRAPH RAG (local search)", query)

    step("STEP 1: ★ Sawal ki entities")
    d = llm_json(f"""List the named entities (people, projects, offices, products, companies) in the question.
Return JSON: {{"entities": ["..."]}}

Question: {query}""")
    entities = [e for e in d.get("entities", []) if isinstance(e, str) and e.strip()]
    print("     ", entities)

    step("STEP 2: ★ Graph mein matching nodes")
    seeds = set()
    for e in entities:
        for node in graph.nodes:
            if e.lower() in node.lower() or (len(node) > 3 and node.lower() in e.lower()):
                seeds.add(node)
    print("     ", sorted(seeds))

    step(f"STEP 3: ★ {hops}-hop padosi (subgraph)")
    nodes, frontier = set(seeds), set(seeds)
    for _ in range(hops):
        neighbours = set()
        for n in frontier:
            neighbours |= set(graph.successors(n)) | set(graph.predecessors(n))
        frontier = neighbours - nodes
        nodes |= neighbours
    facts = [f"{u} --{data['relation']}--> {v}" for u, v, data in graph.subgraph(nodes).edges(data=True)]
    for f in facts[:15]:
        print("      ", f)

    step("STEP 4: LLM (graph facts se)")
    if not facts:
        answer = "I don't know based on the graph."
    else:
        answer = llm("Knowledge-graph facts:\n" + "\n".join(facts) +
                     f"\n\nUsing ONLY these facts, follow the relations step by step and answer: {query}")
    print("\n🤖", answer)
    return answer

graph_rag(Q_MULTIHOP)

### 📌 Yaad rakho
- **Entity resolution** zaroori hai: "Ayesha", "Ayesha Khan", "Ms. Khan" teen alag nodes ban gaye to graph toot jata hai.
- Indexing **mehngi** hai, aur data badle to graph bhi update karna padta hai. Simple fact lookup ke liye vector RAG kaafi hai.
- Behtar hai **hybrid**: graph facts + original chunks dono do, kyun ke triples mein details (tareekhein, shartein) kho jati hain.
- **Interview:** "Graph RAG tab jab data entity-heavy ho, multi-hop sawal aam hon, ya poore corpus ke global/thematic sawal hon."

---
# 8️⃣ Adaptive RAG

### 🧠 Ek line mein
Sawal dekh kar **router** faisla karta hai ke kaunsa pipeline chalana hai. Simple sawal → sasta raasta, complex sawal → mehenga raasta. (Jeong et al., 2024)

### 🗺️ Workflow
```
                     ┌── "direct"    ──► LLM seedha (no retrieval)        → 15% of 240
                     │
Query ──► ★ Router ──┼── "simple"    ──► Advanced RAG (aap ka workflow)   → refund policy
                     │
                     ├── "multi_hop" ──► Multi-hop RAG                    → Falcon lead ka office manager
                     │
                     └── "web"       ──► CRAG (web search)                → Python ka latest version
```
Ye sab se aakhir mein is liye hai kyun ke ye **pichle variants ko hi call karta hai**.

In [19]:
ROUTES = ["direct", "simple", "multi_hop", "web"]

def adaptive_rag(query):
    title("ADAPTIVE RAG", query)

    step("STEP 1: ★ Router")
    d = llm_json(f"""You route questions for a NovaTech (fictional company) assistant. Pick ONE route:
- direct: general knowledge, math, greetings (no documents needed)
- simple: a single fact about NovaTech
- multi_hop: a NovaTech question that needs several chained facts (e.g. "the X of the Y of Z")
- web: public facts or current events outside NovaTech
Return JSON: {{"route": "...", "reason": "..."}}

Question: {query}""")
    route = str(d.get("route", "simple")).strip().lower()
    if route not in ROUTES:
        route = "simple"                       # fallback: ajeeb label aaye to crash na ho
    print(f"      route = {route}  ({d.get('reason', '')})")

    step(f"STEP 2: '{route}' pipeline chalao")
    if route == "direct":
        answer = llm(query)
        print("\n🤖", answer)
    elif route == "simple":
        answer = advanced_rag(query)
    elif route == "multi_hop":
        answer = multi_hop_rag(query)
    else:
        answer = crag(query)
    return answer

for q in [Q_GENERAL, "What is NovaTech's refund policy?", Q_MULTIHOP, Q_OUTSIDE]:
    adaptive_rag(q)
    print()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 ADAPTIVE RAG
❓ What is 15% of 240?

▶ STEP 1: ★ Router
      route = direct  (The question asks for a basic mathematical calculation, which falls under general knowledge and does not require any NovaTech-specific information.)

▶ STEP 2: 'direct' pipeline chalao

🤖 15 % of 240 is **36**.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 ADAPTIVE RAG
❓ What is NovaTech's refund policy?

▶ STEP 1: ★ Router
      route = simple  (The question asks for a single fact about NovaTech's refund policy.)

▶ STEP 2: 'simple' pipeline chalao
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 ADVANCED RAG
❓ What is NovaTech's refund policy?

▶ STEP 1: ★ Query rewrite
      NovaTech refund policy

▶ STEP 2: Query embedding

▶ STEP 3: Vector search (top 5)
     • hr-leave#0                  0.515 | Leave Policy: NovaTech emp

### 📌 Yaad rakho
- Router ki galti sab se mehngi hai: ghalat route = poora pipeline ghalat. Router ka **alag test set** banao (sawal → sahi route).
- Hamesha **fallback route** rakho.
- Traffic zyada ho to LLM router ki jagah chhota classifier (embedding + logistic regression) sasta aur tez hai.
- **Interview:** "Adaptive RAG cost/latency aur accuracy ka balance hai."

---
# 📋 Summary — ek nazar mein

| Variant | Aap ke workflow mein kya badla | LLM calls (approx) | Kab use karo |
|---|---|---|---|
| **Naive** | Rewrite + Rerank hata do | 1 | Baseline, simple FAQ |
| **Advanced** | Rewrite + Rerank (aap ka workflow) | 2 + reranker | Zyada tar production systems |
| **Self-RAG** | + Retrieve? + chunk grading + answer check | 5–8 | Faithfulness critical (legal/medical) |
| **CRAG** | + Evidence score → web search | 5–6 | KB adhoora, bahar ke sawal |
| **Iterative** | Search ↔ "kaafi hai?" loop | 1–3 per round | Missing info pehle se pata nahi |
| **Multi-hop** | + Decompose, har sub-Q par pipeline | hops × 2 + 2 | Chained facts (A→B→C) |
| **Graph RAG** | Vector store → knowledge graph | Indexing: 1 per chunk | Entity-heavy data, global sawal |
| **Adaptive** | + Router shuru mein | +1 | Mixed traffic, cost control |

## 🩺 Failure dekh kar variant chuno (trend dekh kar nahi)

| Baseline mein kya dikha | Pehla ilaj |
|---|---|
| Sahi chunk top-k mein hi nahi | Query rewrite, multi-query, HyDE, hybrid search |
| Sahi chunk aaya magar neeche | **Rerank** |
| Purani/archived info jawab mein | Metadata filter (`status=current`) — sab se sasta fix |
| Docs mein jawab nahi, phir bhi confident jawab | **CRAG** |
| Jawab chunks se match nahi karta (hallucination) | **Self-RAG** checks |
| Jawab kayi docs mein bikhra | **Multi-hop** / **Iterative** / **Graph RAG** |
| Simple sawal bhi slow aur mehenge | **Adaptive** routing |

```
Baseline chalao ─► fail cases dekho ─► categorize ─► usi masle ka ilaj ─► SAME test set par dobara naapo
```

## 🎤 Interview one-liners
- **Self-RAG vs CRAG:** Self-RAG **jawab** ko check karta hai, CRAG **evidence** ko check aur correct karta hai.
- **Iterative vs Multi-hop:** Iterative ek technique (loop), multi-hop ek problem type.
- **Adaptive:** retrieval karna hai ya nahi, aur kaunsi strategy — har query par alag faisla.
- **Graph RAG:** local search = entity ke aas paas ke facts, global search = community summaries par map-reduce.

---
# 🏋️ Practice
1. `SimpleVectorStore.search()` mein `status="current"` filter lagao aur `naive_rag(Q_SIMPLE)` dobara chalao. Kya archived policy hat gayi?
2. `self_rag` mein jab koi chunk relevant na ho, to "I don't know" ke bajaye `web_search()` chalao (Self-RAG + CRAG combo).
3. 8 sawal likho (2 har route ke) aur `adaptive_rag` ka router kitne sahi chunta hai, gino.
4. `DOCS` ki jagah apne store ki FAQ / policies daalo aur dekho baseline kahan fail hota hai.